# LeetCode 1427 — Perform String Shifts bằng Python

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

**Link đề:** https://leetcode.com/problems/perform-string-shifts/
**Độ khó:** Easy · **Chủ đề:** Array, Math, String

```python
class Solution:
    def stringShift(self, s: str, shift: list[list[int]]) -> str:
        ...
```

## Mục tiêu học
1. Hiểu **dịch vòng** (rotation) trái/phải của chuỗi và viết bằng **slicing**.
2. Mô phỏng đúng từng thao tác (baseline) và thấy chi phí phụ thuộc tổng `amount`.
3. Insight: dịch trái `a` rồi dịch phải `b` ⇔ dịch trái `a − b` ⇒ **gộp mọi thao tác** thành một số.
4. Dùng **phép modulo** `% n` để rút gọn (và hiểu `%` của Python với số âm).
5. `collections.deque.rotate`, trace, chứng minh, Time/Space, lỗi hay gặp, bài tập.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

Cho chuỗi `s` (chữ thường) và ma trận `shift`, mỗi dòng `shift[i] = [direction, amount]`:
- `direction = 0` → **dịch trái** (left shift) `amount` lần;
- `direction = 1` → **dịch phải** (right shift) `amount` lần.

**Dịch trái 1 lần:** bỏ ký tự **đầu** và gắn vào **cuối**: `"abc" → "bca"`.
**Dịch phải 1 lần:** bỏ ký tự **cuối** và gắn vào **đầu**: `"abc" → "cab"`.

Thực hiện lần lượt mọi thao tác, trả về chuỗi cuối cùng.

| `s` | `shift` | Output | Diễn biến |
|---|---|---|---|
| `"abc"` | `[[0,1],[1,2]]` | `"cab"` | `abc` →trái 1→ `bca` →phải 2→ `cab` |
| `"abcdefg"` | `[[1,1],[1,1],[0,2],[1,3]]` | `"efgabcd"` | `gabcdef` → `fgabcde` → `abcdefg` → `efgabcd` |

**Constraints (theo hiểu biết):** `1 <= s.length <= 100`, `s` chỉ chữ thường; `1 <= shift.length <= 100`; `shift[i].length == 2`; `direction ∈ {0, 1}`; `0 <= amount <= 100`.

In [1]:
examples = [
    ("abc", [[0, 1], [1, 2]], "cab"),
    ("abcdefg", [[1, 1], [1, 1], [0, 2], [1, 3]], "efgabcd"),
]
for s, shift, expected in examples:
    print(f"s={s!r}, shift={shift} -> kỳ vọng {expected!r}")

s='abc', shift=[[0, 1], [1, 2]] -> kỳ vọng 'cab'
s='abcdefg', shift=[[1, 1], [1, 1], [0, 2], [1, 3]] -> kỳ vọng 'efgabcd'


## 2. Nền tảng Python cần biết

### 2.1. Slicing để dịch 1 lần

- Dịch trái 1: `s[1:] + s[0]` (bỏ đầu, gắn cuối).
- Dịch phải 1: `s[-1] + s[:-1]` (lấy cuối gắn đầu).

`s[-1]` là ký tự cuối; `s[:-1]` là mọi ký tự trừ cuối.

In [2]:
s = "abc"
print("trái 1:", s[1:] + s[0])
print("phải 1:", s[-1] + s[:-1])

trái 1: bca
phải 1: cab


### 2.2. Dịch `k` lần bằng một phép slicing

- Dịch trái `k` (`0 <= k <= n`): `s[k:] + s[:k]` — `k` ký tự đầu chuyển xuống cuối.
- Dịch phải `k`: `s[n − k:] + s[:n − k]` — `k` ký tự cuối chuyển lên đầu.

In [3]:
s = "abcdefg"
n = len(s)
for k in range(0, 4):
    print(f"k={k}: trái -> {s[k:] + s[:k]!r:<10} phải -> {s[n - k:] + s[:n - k]!r}")

k=0: trái -> 'abcdefg'  phải -> 'abcdefg'
k=1: trái -> 'bcdefga'  phải -> 'gabcdef'
k=2: trái -> 'cdefgab'  phải -> 'fgabcde'
k=3: trái -> 'defgabc'  phải -> 'efgabcd'


### 2.3. Dịch `n` lần = không đổi; phép `%` (modulo)

Dịch trái `n` lần đưa mọi ký tự đi hết một vòng ⇒ quay về chuỗi cũ. Vậy dịch trái `k` ⇔ dịch trái `k % n`.

Python: `%` luôn cho kết quả **cùng dấu với số chia** — với `n > 0` thì `k % n ∈ [0, n − 1]` **kể cả khi `k` âm**. Ví dụ `-3 % 7 = 4`. (Khác C/Java, nơi `-3 % 7 = -3`.)

In [4]:
n = 7
for k in [3, 7, 10, 0, -3, -10]:
    print(f"{k:>4} % {n} = {k % n}")

   3 % 7 = 3
   7 % 7 = 0
  10 % 7 = 3
   0 % 7 = 0
  -3 % 7 = 4
 -10 % 7 = 4


### 2.4. Dịch phải `k` ⇔ dịch trái `n − k` ⇔ dịch trái `−k`

Vì dịch trái rồi dịch phải cùng số lần thì triệt tiêu, nên **dịch phải là dịch trái âm**. Nhờ `%`, dịch trái `−k` ⇔ dịch trái `(−k) % n = n − k` (khi `0 < k < n`).

In [5]:
s = "abcdefg"
n = len(s)
k = 3
right = s[n - k:] + s[:n - k]
left_neg = s[(-k) % n:] + s[:(-k) % n]
print(right, left_neg, right == left_neg)

efgabcd efgabcd True


### 2.5. Gán bộ khi duyệt list 2 phần tử

`for direction, amount in shift:` tách mỗi dòng `[d, a]` thành hai biến (tuple/list unpacking).

In [6]:
for direction, amount in [[0, 1], [1, 2]]:
    print("direction =", direction, "| amount =", amount, "|", "trái" if direction == 0 else "phải")

direction = 0 | amount = 1 | trái
direction = 1 | amount = 2 | phải


## 3. Cách trực giác (baseline) — mô phỏng từng lần dịch 1 ký tự

Đúng như đề: với mỗi thao tác, lặp `amount` lần, mỗi lần dịch 1 ký tự.

In [7]:
def string_shift_naive(s: str, shift: list[list[int]]) -> str:
    for direction, amount in shift:
        for _ in range(amount):
            if direction == 0:
                s = s[1:] + s[0]       # trái 1
            else:
                s = s[-1] + s[:-1]     # phải 1
    return s

for s, shift, expected in examples:
    print(repr(s), shift, "->", repr(string_shift_naive(s, shift)), "| kỳ vọng", repr(expected))

'abc' [[0, 1], [1, 2]] -> 'cab' | kỳ vọng 'cab'
'abcdefg' [[1, 1], [1, 1], [0, 2], [1, 3]] -> 'efgabcd' | kỳ vọng 'efgabcd'


### 3.1. Nút thắt

Mỗi lần dịch 1 ký tự tạo chuỗi mới `O(n)`. Tổng số lần dịch = `Σ amount` ≤ `100 · 100 = 10^4` ⇒ `O(n · Σ amount)` ≈ `10^6`. Chạy được, nhưng:
- Lãng phí: dịch `n` lần liên tiếp chẳng thay đổi gì.
- Lãng phí: dịch trái rồi dịch phải **triệt tiêu** nhau.
- Nếu `amount` tới `10^9` thì không thể mô phỏng.

In [8]:
s = "abcdefg"
t = s
for _ in range(len(s)):
    t = t[1:] + t[0]
print(f"dịch trái {len(s)} lần: {t!r} == {s!r}? {t == s}")
t = s[1:] + s[0]
t = t[-1] + t[:-1]
print(f"trái 1 rồi phải 1: {t!r} == {s!r}? {t == s}")

dịch trái 7 lần: 'abcdefg' == 'abcdefg'? True
trái 1 rồi phải 1: 'abcdefg' == 'abcdefg'? True


## 4. Insight — gộp tất cả thành **một** lần dịch

Mọi thao tác đều là dịch vòng trên **cùng một vòng tròn** ký tự. Dịch trái `a` rồi trái `b` = trái `a + b`; dịch phải = dịch trái âm.

⇒ `net_left = Σ (amount nếu direction = 0, ngược lại −amount)`.
⇒ `k = net_left % n`.
⇒ đáp án `= s[k:] + s[:k]`.

Ví dụ 2: `net_left = −1 −1 +2 −3 = −3`; `k = −3 % 7 = 4`; `"abcdefg"[4:] + "abcdefg"[:4] = "efg" + "abcd"` ✔.

In [9]:
s, shift = "abcdefg", [[1, 1], [1, 1], [0, 2], [1, 3]]
net_left = 0
for d, a in shift:
    net_left += a if d == 0 else -a
    print(f"[{d}, {a}] -> net_left = {net_left}")
k = net_left % len(s)
print("k =", k, "->", s[k:] + s[:k])

[1, 1] -> net_left = -1
[1, 1] -> net_left = -2
[0, 2] -> net_left = 0
[1, 3] -> net_left = -3
k = 4 -> efgabcd


## 5. Lời giải chính — giảng từng đoạn

**Đoạn 1:** `net = 0`; với mỗi `[d, a]`: `net += a` nếu `d == 0`, ngược lại `net -= a`.

**Đoạn 2:** `k = net % len(s)` — đưa về `[0, n − 1]`, xử lý luôn số âm.

**Đoạn 3:** `return s[k:] + s[:k]`.

In [10]:
def string_shift_net(s: str, shift: list[list[int]]) -> str:
    net = 0                                    # Đoạn 1
    for direction, amount in shift:
        if direction == 0:
            net += amount
        else:
            net -= amount
    k = net % len(s)                           # Đoạn 2
    return s[k:] + s[:k]                       # Đoạn 3

for s, shift, expected in examples:
    print(repr(s), "->", repr(string_shift_net(s, shift)), "| kỳ vọng", repr(expected))

'abc' -> 'cab' | kỳ vọng 'cab'
'abcdefg' -> 'efgabcd' | kỳ vọng 'efgabcd'


### 5.1. Bản gọn với `sum` + biểu thức điều kiện

In [11]:
def string_shift_sum(s: str, shift: list[list[int]]) -> str:
    k = sum(a if d == 0 else -a for d, a in shift) % len(s)
    return s[k:] + s[:k]

print([string_shift_sum(s, sh) for s, sh, _ in examples])

['cab', 'efgabcd']


### 5.2. Mẹo `(-1) ** d` hoặc `1 - 2 * d`

`d = 0` → hệ số `+1` (trái); `d = 1` → `−1` (phải). Có thể viết `a * (1 - 2 * d)`. Ngắn nhưng khó đọc hơn — dùng khi đã hiểu.

In [12]:
for d in [0, 1]:
    print(f"d={d}: 1 - 2*d = {1 - 2 * d}, (-1)**d = {(-1) ** d}")

d=0: 1 - 2*d = 1, (-1)**d = 1
d=1: 1 - 2*d = -1, (-1)**d = -1


## 6. Cách 3 — mô phỏng mỗi thao tác bằng một slicing (rút gọn từng bước)

Trung gian giữa baseline và lời giải chính: vẫn thực hiện **từng thao tác**, nhưng mỗi thao tác chỉ 1 phép slicing với `amount % n`. Chi phí `O(m · n)`.

In [13]:
def string_shift_per_op(s: str, shift: list[list[int]]) -> str:
    n = len(s)
    for d, a in shift:
        a %= n
        if d == 0:
            s = s[a:] + s[:a]
        else:
            s = s[n - a:] + s[:n - a]
    return s

print([string_shift_per_op(s, sh) for s, sh, _ in examples])

['cab', 'efgabcd']


## 7. Cách 4 — `collections.deque.rotate`

`deque` (double-ended queue) thêm/bớt ở **hai đầu** trong `O(1)`. `dq.rotate(r)` xoay **phải** `r` bước (`r` âm → xoay trái). Ghép lại bằng `"".join(dq)`.

In [14]:
from collections import deque

dq = deque("abcde")
dq.rotate(2)
print("rotate(2) :", "".join(dq))
dq.rotate(-2)
print("rotate(-2):", "".join(dq))

def string_shift_deque(s: str, shift: list[list[int]]) -> str:
    dq = deque(s)
    net_right = sum(a if d == 1 else -a for d, a in shift)
    dq.rotate(net_right % len(s))
    return "".join(dq)

print([string_shift_deque(s, sh) for s, sh, _ in examples])

rotate(2) : deabc
rotate(-2): abcde
['cab', 'efgabcd']


## 8. TRACE từng bước

Trace lời giải chính với `s = "abcdefg"`, `shift = [[1,1],[1,1],[0,2],[1,3]]`, kèm chuỗi thật sau mỗi thao tác để đối chiếu:

| bước | [d, a] | ý nghĩa | net_left | chuỗi thật (mô phỏng) | `s[net % 7:] + s[:net % 7]` |
|---:|---|---|---:|---|---|
| 1 | [1,1] | phải 1 | −1 | `gabcdef` | `k=6` → `gabcdef` |
| 2 | [1,1] | phải 1 | −2 | `fgabcde` | `k=5` → `fgabcde` |
| 3 | [0,2] | trái 2 | 0 | `abcdefg` | `k=0` → `abcdefg` |
| 4 | [1,3] | phải 3 | −3 | `efgabcd` | `k=4` → `efgabcd` |

Cột cuối luôn khớp cột "chuỗi thật" — đó chính là bất biến của thuật toán.

In [15]:
def trace_string_shift(s: str, shift: list[list[int]]) -> str:
    n = len(s)
    real = s
    net = 0
    print(f"s = {s!r}, n = {n}")
    for step, (d, a) in enumerate(shift, start=1):
        net += a if d == 0 else -a
        for _ in range(a):                               # mô phỏng thật để đối chiếu
            real = real[1:] + real[0] if d == 0 else real[-1] + real[:-1]
        k = net % n
        via_net = s[k:] + s[:k]
        print(f"  bước {step}: [{d}, {a}] {'trái' if d == 0 else 'phải'} {a} | net={net:>3}, k={k} "
              f"| thật={real!r} | công thức={via_net!r} | khớp={real == via_net}")
    return s[net % n:] + s[:net % n]

print("=>", repr(trace_string_shift("abcdefg", [[1, 1], [1, 1], [0, 2], [1, 3]])))

s = 'abcdefg', n = 7
  bước 1: [1, 1] phải 1 | net= -1, k=6 | thật='gabcdef' | công thức='gabcdef' | khớp=True
  bước 2: [1, 1] phải 1 | net= -2, k=5 | thật='fgabcde' | công thức='fgabcde' | khớp=True
  bước 3: [0, 2] trái 2 | net=  0, k=0 | thật='abcdefg' | công thức='abcdefg' | khớp=True
  bước 4: [1, 3] phải 3 | net= -3, k=4 | thật='efgabcd' | công thức='efgabcd' | khớp=True
=> 'efgabcd'


`for step, (d, a) in enumerate(shift, start=1)`: `enumerate` trả `(chỉ_số, phần_tử)`, và phần tử `[d, a]` được tách tiếp bằng `(d, a)`.

Trace thêm một trường hợp `amount` lớn hơn `n` (rút gọn bằng `%`):

In [16]:
print("=>", repr(trace_string_shift("abc", [[0, 7], [1, 100]])))
print("mô phỏng:", repr(string_shift_naive("abc", [[0, 7], [1, 100]])))

s = 'abc', n = 3
  bước 1: [0, 7] trái 7 | net=  7, k=1 | thật='bca' | công thức='bca' | khớp=True
  bước 2: [1, 100] phải 100 | net=-93, k=0 | thật='abc' | công thức='abc' | khớp=True
=> 'abc'
mô phỏng: 'abc'


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def stringShift(self, s: str, shift: list[list[int]]) -> str:
        net = 0
        for direction, amount in shift:
            net += amount if direction == 0 else -amount
        k = net % len(s)
        return s[k:] + s[:k]

print([Solution().stringShift(s, sh) for s, sh, _ in examples])

['cab', 'efgabcd']


Trên LeetCode chữ ký gốc thường dùng `List[List[int]]` (`from typing import List`) — tương đương `list[list[int]]`.

### Bản deque

In [18]:
class SolutionDeque:
    def stringShift(self, s: str, shift: list[list[int]]) -> str:
        dq = deque(s)
        for d, a in shift:
            dq.rotate(a if d == 1 else -a)
        return "".join(dq)

print([SolutionDeque().stringShift(s, sh) for s, sh, _ in examples])

['cab', 'efgabcd']


### Bảng so sánh

Gọi `n = len(s)`, `m = len(shift)`, `A = Σ amount`.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Mô phỏng từng ký tự | dịch 1 lần × amount | `O(A · n)` | `O(n)` | đúng đề, chậm |
| 2. Gộp net + 1 slicing | `k = net % n` | `O(m + n)` | `O(n)` | tối ưu |
| 3. Slicing mỗi thao tác | `a % n` từng bước | `O(m · n)` | `O(n)` | trung gian |
| 4. `deque.rotate` | xoay vòng | `O(m · n)` xấu nhất (hoặc `O(m + n)` nếu gộp) | `O(n)` | thư viện chuẩn |

## 10. Vì sao đúng?

Đánh số ký tự theo vị trí `0..n−1`. **Dịch trái 1** đưa ký tự ở vị trí `p` về `(p − 1) mod n`; **dịch phải 1** đưa về `(p + 1) mod n`.

Sau mọi thao tác, ký tự ban đầu ở `p` nằm tại `(p − net_left) mod n`, vì phép cộng modulo **giao hoán và kết hợp** — thứ tự các thao tác không quan trọng, chỉ tổng có ý nghĩa.

Vậy ký tự ở vị trí `q` của kết quả là ký tự ban đầu ở `(q + net_left) mod n = (q + k) mod n`, tức `s[k], s[k+1], ..., s[n−1], s[0], ..., s[k−1]` = `s[k:] + s[:k]`. ∎

**Bất biến vòng lặp:** sau khi xử lý `i` thao tác đầu, chuỗi thật = `s[net % n:] + s[:net % n]` (đã kiểm chứng ở trace).

## 11. Time / Space

- Lời giải chính: duyệt `shift` `O(m)` + tạo chuỗi kết quả `O(n)` ⇒ **`O(m + n)`**; bộ nhớ `O(n)` cho chuỗi mới (chuỗi Python bất biến nên không thể tránh).
- Mô phỏng: `O(A · n)` với `A ≤ 10^4` theo constraint.
- `deque.rotate(r)`: `O(min(r, n − r))` mỗi lần.

In [19]:
import random, string, time
random.seed(1)
s_big = "".join(random.choice(string.ascii_lowercase) for _ in range(100))
sh_big = [[random.randint(0, 1), random.randint(0, 100)] for _ in range(100)]
for f in [string_shift_naive, string_shift_per_op, string_shift_net, string_shift_deque]:
    t0 = time.perf_counter()
    for _ in range(50):
        r = f(s_big, sh_big)
    print(f"{f.__name__:<22} {r[:12]}... ({1000 * (time.perf_counter() - t0):7.2f} ms / 50 lần)")

string_shift_naive     anrudfuxjdxk... (  25.05 ms / 50 lần)
string_shift_per_op    anrudfuxjdxk... (   1.38 ms / 50 lần)
string_shift_net       anrudfuxjdxk... (   0.21 ms / 50 lần)
string_shift_deque     anrudfuxjdxk... (   0.72 ms / 50 lần)


## 12. Bộ test

Mẫu; chuỗi 1 ký tự; `amount = 0`; `amount` là bội của `n`; trái và phải triệt tiêu; `amount` > `n`; và **đối chiếu ngẫu nhiên** mọi cách với mô phỏng.

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS -> {actual!r}")

sol = Solution()
check("ví dụ 1", sol.stringShift("abc", [[0, 1], [1, 2]]), "cab")
check("ví dụ 2", sol.stringShift("abcdefg", [[1, 1], [1, 1], [0, 2], [1, 3]]), "efgabcd")
check("1 ký tự", sol.stringShift("z", [[0, 5], [1, 3]]), "z")
check("amount = 0", sol.stringShift("abc", [[0, 0], [1, 0]]), "abc")
check("bội của n", sol.stringShift("abcd", [[0, 8]]), "abcd")
check("triệt tiêu", sol.stringShift("abcd", [[0, 3], [1, 3]]), "abcd")
check("amount > n (trái)", sol.stringShift("abc", [[0, 4]]), "bca")
check("amount > n (phải)", sol.stringShift("abc", [[1, 5]]), "bca")

ví dụ 1                PASS -> 'cab'
ví dụ 2                PASS -> 'efgabcd'
1 ký tự                PASS -> 'z'
amount = 0             PASS -> 'abc'
bội của n              PASS -> 'abcd'
triệt tiêu             PASS -> 'abcd'
amount > n (trái)      PASS -> 'bca'
amount > n (phải)      PASS -> 'bca'


In [21]:
methods = [string_shift_net, string_shift_sum, string_shift_per_op, string_shift_deque,
           Solution().stringShift, SolutionDeque().stringShift]
random.seed(1427)
for _ in range(2000):
    s = "".join(random.choice("abcde") for _ in range(random.randint(1, 8)))
    shift = [[random.randint(0, 1), random.randint(0, 20)] for _ in range(random.randint(1, 6))]
    expected = string_shift_naive(s, shift)
    for f in methods:
        assert f(s, shift) == expected, (f, s, shift)
print("Mọi cách khớp mô phỏng trên 2000 bộ ngẫu nhiên")

Mọi cách khớp mô phỏng trên 2000 bộ ngẫu nhiên


## 13. Lỗi người mới hay gặp

### Lỗi 1: Nhầm chiều dịch (đảo trái/phải)

In [22]:
def wrong_direction(s, shift):
    k = sum(a if d == 1 else -a for d, a in shift) % len(s)   # SAI: coi 1 là trái
    return s[k:] + s[:k]

print("sai:", wrong_direction("abc", [[0, 1]]), "| đúng:", string_shift_net("abc", [[0, 1]]))

sai: cab | đúng: bca


### Lỗi 2: Quên `% n` khi `amount > n` dùng slicing trực tiếp

`s[a:] + s[:a]` với `a > n`: `s[a:]` rỗng, `s[:a]` là cả chuỗi ⇒ trả về nguyên `s` — sai.

In [23]:
s, a = "abc", 4
print("không %:", repr(s[a:] + s[:a]), "| có %:", repr(s[a % 3:] + s[:a % 3]))

không %: 'abc' | có %: 'bca'


### Lỗi 3: Dịch phải bằng `s[-k:] + s[:-k]` khi `k = 0`

`s[-0:]` = `s[0:]` = cả chuỗi, `s[:-0]` = `s[:0]` = rỗng ⇒ vẫn ra `s` (may mắn đúng). Nhưng nếu viết `s[-k:]` với `k` chưa `% n` (ví dụ `k = n + 1`) thì lỗi logic. Cách an toàn: luôn đổi về **dịch trái** với `k = net % n` rồi `s[k:] + s[:k]`.

In [24]:
s = "abcd"
for k in [0, 1, 5]:
    print(f"k={k}: s[-k:]+s[:-k] = {s[-k:] + s[:-k]!r} | đúng (phải k) = {string_shift_net(s, [[1, k]])!r}")

k=0: s[-k:]+s[:-k] = 'abcd' | đúng (phải k) = 'abcd'
k=1: s[-k:]+s[:-k] = 'dabc' | đúng (phải k) = 'dabc'
k=5: s[-k:]+s[:-k] = 'abcd' | đúng (phải k) = 'dabc'


### Lỗi 4: Tưởng `%` của Python giống C/Java với số âm

Một số bạn viết `k = net % n; if k < 0: k += n` — thừa trong Python (vô hại). Nhưng nếu **chuyển sang C++/Java** mà bỏ dòng đó thì sai. Ngược lại, viết `abs(net) % n` là **sai** vì mất chiều.

In [25]:
def wrong_abs(s, shift):
    net = sum(a if d == 0 else -a for d, a in shift)
    k = abs(net) % len(s)                  # SAI: mất dấu -> mất chiều dịch
    return s[k:] + s[:k]

print("sai:", wrong_abs("abcdefg", [[1, 3]]), "| đúng:", string_shift_net("abcdefg", [[1, 3]]))

sai: defgabc | đúng: efgabcd


## 14. Mẹo nhận dạng pattern

- "Xoay/dịch vòng" nhiều lần ⇒ gộp thành **một** độ dịch ròng, rồi `% n`.
- Dịch phải `k` = dịch trái `−k` = dịch trái `n − k`.
- Xoay chuỗi/mảng ⇒ `s[k:] + s[:k]`; hoặc `deque.rotate`; hoặc kỹ thuật **ba lần đảo ngược** (reverse) khi cần làm tại chỗ với mảng.
- Thao tác **giao hoán** (thứ tự không quan trọng) ⇒ cộng dồn rồi áp dụng một lần.

## 15. Biến thể

1. LeetCode 189 *Rotate Array*: xoay mảng phải `k` bước **tại chỗ** — kỹ thuật 3 lần đảo ngược.
2. LeetCode 796 *Rotate String*: kiểm tra `goal` có phải một phép xoay của `s` — mẹo `goal in s + s`.
3. LeetCode 848 *Shifting Letters*: dịch **giá trị chữ cái** (không phải vị trí) — dùng suffix sum.
4. Trả về **mọi** chuỗi trung gian sau từng thao tác.

In [26]:
def rotate_array_inplace(nums: list[int], k: int) -> None:      # LeetCode 189
    n = len(nums)
    k %= n
    def reverse(i, j):
        while i < j:
            nums[i], nums[j] = nums[j], nums[i]
            i, j = i + 1, j - 1
    reverse(0, n - 1)        # đảo cả mảng
    reverse(0, k - 1)        # đảo k phần tử đầu
    reverse(k, n - 1)        # đảo phần còn lại

def rotate_string(s: str, goal: str) -> bool:                    # LeetCode 796
    return len(s) == len(goal) and goal in s + s

a = [1, 2, 3, 4, 5, 6, 7]
rotate_array_inplace(a, 3)
print(a)
print(rotate_string("abcde", "cdeab"), rotate_string("abcde", "abced"))

[5, 6, 7, 1, 2, 3, 4]
True False


`nums[i], nums[j] = nums[j], nums[i]` hoán đổi hai phần tử trong một dòng. Vì sao 3 lần đảo đúng: xoay phải `k` nghĩa là `k` phần tử cuối lên đầu; đảo cả mảng đưa chúng lên đầu (nhưng ngược thứ tự), hai lần đảo cục bộ sửa lại thứ tự.

## 16. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** `s = "hello"`, `shift = [[1, 2], [0, 4], [1, 1]]`. Tính `net_left`, `k`, kết quả.

**Bài 2 (đoán output):** `-7 % 5`, `"python"[2:] + "python"[:2]`, `"abc"[5:]`, `deque([1, 2, 3])` sau `rotate(1)`.

**Bài 3 (điền chỗ trống):** `exercise_string_shift(s, shift)` — gộp net, `%`, slicing.

**Bài 4 (tự viết):** `exercise_rotate_right(s, k)` — dịch phải `k` (có thể `k > n`) chỉ bằng slicing.

**Bài 5 (biến thể):** `exercise_all_states(s, shift)` — list chuỗi **sau mỗi thao tác** (độ dài `len(shift)`), mỗi bước `O(n)`.

**Bài 6 (mini challenge):** `exercise_min_rotation(s)` — trong mọi phép xoay của `s`, trả về chuỗi **nhỏ nhất theo thứ tự từ điển** (`O(n²)` là đủ).

In [27]:
def exercise_string_shift(s: str, shift: list[list[int]]) -> str:
    # TODO 1: net = 0, cộng/trừ theo direction
    # TODO 2: k = net % len(s)
    # TODO 3: return s[k:] + s[:k]
    pass

def exercise_rotate_right(s: str, k: int) -> str:
    # TODO
    pass

# assert exercise_string_shift("abc", [[0, 1], [1, 2]]) == "cab"
# assert exercise_rotate_right("abcde", 7) == "deabc"

In [28]:
def exercise_all_states(s: str, shift: list[list[int]]) -> list[str]:
    # TODO
    pass

def exercise_min_rotation(s: str) -> str:
    # TODO
    pass

# assert exercise_all_states("abc", [[0, 1], [1, 2]]) == ["bca", "cab"]
# assert exercise_min_rotation("cba") == "acb"

## 17. Gợi ý 3 mức

**Bài 1** — (1) `net = −2 + 4 − 1`. (2) `k = net % 5`. (3) `s[k:] + s[:k]`.

**Bài 3** — (1) `net += a if d == 0 else -a`. (2) `%` xử lý âm. (3) slicing.

**Bài 4** — (1) Dịch phải `k` = dịch trái `−k`. (2) `j = (-k) % n`. (3) `s[j:] + s[:j]`.

**Bài 5** — (1) Giữ `net` cộng dồn. (2) Sau mỗi thao tác tính `k = net % n`. (3) Append `s[k:] + s[:k]` (từ chuỗi **gốc** `s`).

**Bài 6** — (1) Có `n` phép xoay: `s[i:] + s[:i]`. (2) `min(...)` trên chuỗi so theo từ điển. (3) Hoặc mẹo: mọi phép xoay là chuỗi con độ dài `n` của `s + s`.

## 18. Đáp án tham khảo

**Bài 1:** `net = −2 + 4 − 1 = 1`, `k = 1` ⇒ `"elloh"`.

**Bài 2:** `3`, `"thonpy"`, `""`, `deque([3, 1, 2])`.

In [29]:
def answer_string_shift(s, shift):
    net = 0
    for d, a in shift:
        net += a if d == 0 else -a
    k = net % len(s)
    return s[k:] + s[:k]

def answer_rotate_right(s, k):
    j = (-k) % len(s)
    return s[j:] + s[:j]

def answer_all_states(s, shift):
    n, net, out = len(s), 0, []
    for d, a in shift:
        net += a if d == 0 else -a
        k = net % n
        out.append(s[k:] + s[:k])
    return out

def answer_min_rotation(s):
    return min(s[i:] + s[:i] for i in range(len(s)))

assert answer_string_shift("hello", [[1, 2], [0, 4], [1, 1]]) == "elloh"
dq = deque([1, 2, 3]); dq.rotate(1)
assert (-7 % 5, "python"[2:] + "python"[:2], "abc"[5:], dq) == (3, "thonpy", "", deque([3, 1, 2]))
assert answer_string_shift("abc", [[0, 1], [1, 2]]) == "cab"
assert answer_rotate_right("abcde", 7) == "deabc"
assert answer_all_states("abc", [[0, 1], [1, 2]]) == ["bca", "cab"]
assert answer_min_rotation("cba") == "acb"
random.seed(4)
for _ in range(500):
    s = "".join(random.choice("abc") for _ in range(random.randint(1, 7)))
    shift = [[random.randint(0, 1), random.randint(0, 15)] for _ in range(random.randint(1, 5))]
    states = answer_all_states(s, shift)
    assert states[-1] == string_shift_naive(s, shift) == answer_string_shift(s, shift)
    for i in range(len(shift)):
        assert states[i] == string_shift_naive(s, shift[:i + 1])
    k = random.randint(0, 20)
    assert answer_rotate_right(s, k) == string_shift_naive(s, [[1, k]])
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def stringShift(self, s: str, shift: list[list[int]]) -> str:
        net = sum(a if d == 0 else -a for d, a in shift)   # trái +, phải -
        k = net % len(s)                                   # Python: luôn 0..n-1
        return s[k:] + s[:k]                               # dịch trái k
```

**Khung tư duy "xoay vòng nhiều lần":** quy về một chiều → cộng dồn → `% n` → một phép xoay.

**Checklist:**
- [ ] `direction = 0` là **trái**, `1` là **phải**.
- [ ] Luôn `% len(s)` trước khi slicing.
- [ ] Không dùng `abs(net)` (mất chiều).
- [ ] Dịch trái `k`: `s[k:] + s[:k]`.
- [ ] (Premium) Đối chiếu lại đề/constraints nếu có tài khoản.